# 基于LoRA的BERT中文文本分类 - 外卖评论二分类

## 项目说明
本项目使用 **LoRA (Low-Rank Adaptation)** 技术对本地 bert-base-chinese 模型进行高效微调，实现外卖评论的情感分类任务。

## LoRA 优势
1. **参数效率**：只训练少量参数（通常<1%），大幅降低显存需求
2. **训练速度**：训练速度更快，适合资源受限的环境
3. **模型保存**：只需保存LoRA权重（几MB），而不是整个模型（几百MB）
4. **多任务切换**：可以为不同任务训练不同的LoRA权重，共享基础模型

## 模型文件夹位置
模型保存在：`e:\BertTunning\bert-base-chinese\`

### 需要下载的文件：
- config.json（模型配置文件）
- pytorch_model.bin（预训练模型权重）
- vocab.txt（词汇表）
- tokenizer_config.json（分词器配置）

### 下载地址：
https://huggingface.co/bert-base-chinese/tree/main

---

## 项目结构
1. **环境配置**：导入必要的库和设置路径
2. **配置参数**：设置超参数和LoRA配置
3. **随机种子设置**：确保结果可复现
4. **数据准备**：封装的数据加载函数
5. **模型定义**：定义数据集类和带LoRA的BERT分类模型
6. **训练与评估**：包含最佳模型保存的完整训练流程

In [2]:
"""
==========================================
第一部分：导入必要的库
==========================================
本部分导入项目所需的所有Python库和模块
"""

# PyTorch相关库
import torch                    # PyTorch核心库，用于张量操作和自动求导
import torch.nn as nn           # 神经网络模块，用于定义模型层
from torch.utils.data import DataLoader, Dataset  # 数据加载器，用于批量处理数据

# 数据处理库
import pandas as pd             # 用于读取和处理CSV数据
import numpy as np              # 用于数值计算和数组操作

# 系统库
import os                       # 用于文件路径操作和检查文件是否存在
import random                   # 用于设置随机种子
import time                     # 用于训练计时

# 评估指标
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Transformers库（Hugging Face）
from transformers import BertTokenizer, BertModel, BertForSequenceClassification  # BERT分词器和模型

# PEFT库（Parameter-Efficient Fine-Tuning）
from peft import LoraConfig, get_peft_model, TaskType  # LoRA配置和模型

# 优化器和工具
from torch.optim import AdamW   # AdamW优化器，用于模型参数更新
from tqdm import tqdm           # 进度条显示，用于训练过程可视化

# 数据分割工具
from sklearn.model_selection import train_test_split  # 用于划分训练集、验证集和测试集

print("✅ 所有库导入完成")

✅ 所有库导入完成

💡 提示：如果提示 'peft' 模块未找到，请运行：pip install peft


In [ ]:
"""
==========================================
第二部分：配置参数
==========================================
设置模型路径、超参数、LoRA配置等
"""

# ========== 模型路径配置 ==========
# 设置本地 BERT 模型路径（离线模式）
BERT_MODEL_PATH = r'../bert-base-chinese'
DATA_DIR = '../waimai.csv'

# 检查模型文件夹是否存在
if not os.path.exists(BERT_MODEL_PATH):
    print(f"⚠️  警告：模型文件夹不存在: {BERT_MODEL_PATH}")
else:
    print(f"✅ 模型路径检查通过: {BERT_MODEL_PATH}")

# ========== 超参数配置 ==========
# 训练相关参数
EPOCHS = 5              # 训练轮数（LoRA可以训练更多轮次）
LEARNING_RATE = 3e-4    #
BATCH_SIZE = 32         # 批次大小（根据GPU内存调整）

# 模型相关参数
MAX_LENGTH = 256        # 文本最大长度（BERT最大支持512）
DROPOUT = 0.5
NUM_CLASSES = 2         # 分类类别数（二分类：正面/负面）

# 数据分割比例
TRAIN_RATIO = 0.8       # 训练集比例
VAL_RATIO = 0.1        # 验证集比例
TEST_RATIO = 0.1       # 测试集比例

# ========== LoRA 配置 ==========
LORA_R = 8              # LoRA秩（rank），控制低秩矩阵的维度，通常取4-16
LORA_ALPHA = 16         # LoRA缩放参数，通常设置为r的2倍
LORA_DROPOUT = 0.1      # LoRA的Dropout比率
# LoRA目标模块：指定要应用LoRA的模块
# BERT中通常对query和value的投影矩阵应用LoRA
LORA_TARGET_MODULES = ["query", "value","key","dense"]  # 也可以添加 "key", "dense" 等

#优化器
WEIGHT_DECAY=0.01

#是否冻结主干模型
FREEZE_BERT=False
# ========== 随机种子和模型保存配置 ==========
RANDOM_SEED = 42        # 随机种子，确保结果可复现
SAVE_PATH = './bert_lora_checkpoint'  # 模型保存路径

In [ ]:
"""
==========================================
第三部分：设置随机种子
==========================================
确保实验结果可复现
"""

def setup_seed(seed):
    """
    设置随机种子，确保实验结果可复现
    
    参数:
        seed: 随机种子值
    """
    torch.manual_seed(seed)              # 设置PyTorch的随机种子
    torch.cuda.manual_seed_all(seed)     # 设置所有GPU的随机种子
    np.random.seed(seed)                 # 设置NumPy的随机种子
    random.seed(seed)                    # 设置Python内置random的随机种子
    torch.backends.cudnn.deterministic = True  # 确保CUDA操作可复现

# 应用随机种子
setup_seed(RANDOM_SEED)

print("✅ 随机种子设置完成，实验结果可复现")

In [ ]:
"""
==========================================
第四部分：初始化分词器
==========================================
加载BERT分词器
"""

# 从本地路径加载BERT分词器
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_PATH)

print("✅ 分词器加载完成")
print(f"   - 词汇表大小: {len(tokenizer.vocab)}")
print(f"   - 特殊token示例: [CLS]={tokenizer.cls_token}, [SEP]={tokenizer.sep_token}, [PAD]={tokenizer.pad_token}")

In [6]:
"""
==========================================
第五部分：定义数据集类
==========================================
自定义Dataset类，用于将原始数据转换为模型可用的格式
"""

class TextDataset(Dataset):
    """
    文本分类数据集类
    
    功能：
    1. 将原始文本数据转换为BERT可接受的格式
    2. 对文本进行分词、padding和截断处理
    3. 返回处理后的文本和对应的标签
    """
    
    def __init__(self, df):
        """
        初始化数据集
        
        参数:
            df: pandas DataFrame，包含 'review' 和 'label' 列
        """
        # 将标签转换为numpy数组
        self.labels = df['label'].astype(int).values
        
        # 对每个文本进行分词处理
        self.texts = [
            tokenizer(
                text, 
                padding='max_length', 
                max_length=MAX_LENGTH, 
                truncation=True,
                return_tensors="pt"
            ) 
            for text in df['review']
        ]
        
    def __len__(self):
        """返回数据集大小"""
        return len(self.labels)
    
    def __getitem__(self, idx):
        """
        获取单个数据样本
        
        参数:
            idx: 数据索引
            
        返回:
            batch_texts: 处理后的文本（包含input_ids和attention_mask）
            batch_y: 对应的标签
        """
        batch_texts = self.get_batch_texts(idx)
        batch_y = self.get_batch_labels(idx)
        return batch_texts, batch_y
    
    def classes(self):
        """返回所有标签"""
        return self.labels
    
    def get_batch_labels(self, idx):
        """
        获取标签并转换为Long类型张量
        """
        return torch.tensor(self.labels[idx], dtype=torch.long)

    def get_batch_texts(self, idx):
        """获取处理后的文本数据"""
        return self.texts[idx]

print("✅ 数据集类定义完成")

✅ 数据集类定义完成


In [7]:
"""
==========================================
第六部分：数据加载函数
==========================================
封装数据加载和分割逻辑
"""

def GenerateData(mode='train'):
    """
    加载和预处理数据，并返回指定模式的数据集
    
    参数:
        mode: 数据模式，可选 'train'、'val' 或 'test'
    
    返回:
        TextDataset: 处理后的数据集对象
    """
    # 读取CSV数据文件
    data_path = DATA_DIR
    df = pd.read_csv(data_path)
    
    # 查看数据基本信息
    if mode == 'train':
        print("=" * 50)
        print("数据基本信息")
        print("=" * 50)
        print(f"数据总量: {len(df)}")
        print(f"\n标签分布:")
        print(df['label'].value_counts())
        print(f"\n数据示例（前3条）:")
        print(df.head(3))
    
    # 数据分割
    train_df, temp_df = train_test_split(
        df,
        test_size=0.3,
        stratify=df['label'],
        random_state=RANDOM_SEED
    )
    
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df['label'],
        random_state=RANDOM_SEED
    )
    
    # 根据模式返回对应的数据集
    if mode == 'train':
        print("\n" + "=" * 50)
        print("数据分割结果")
        print("=" * 50)
        print(f"训练集大小: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
        print(f"验证集大小: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
        print(f"测试集大小: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")
        print("\n✅ 数据加载和预处理完成")
        return TextDataset(train_df)
    elif mode == 'val':
        return TextDataset(val_df)
    elif mode == 'test':
        return TextDataset(test_df)
    else:
        raise ValueError(f"不支持的模式: {mode}，请使用 'train'、'val' 或 'test'")

print("✅ 数据加载函数定义完成")

✅ 数据加载函数定义完成


In [8]:
"""
==========================================
第七部分：初始化BERT模型并应用LoRA
==========================================
使用PEFT库为BERT模型添加LoRA适配器
"""

# 加载预训练的BERT分类模型
# 使用 BertForSequenceClassification 自动添加分类头
model = BertForSequenceClassification.from_pretrained(
    BERT_MODEL_PATH,
    num_labels=NUM_CLASSES
)

print("✅ BERT基础模型加载完成")
print(f"   - 原始模型参数数量: {sum(p.numel() for p in model.parameters()):,}")

# 配置LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # 序列分类任务
    r=LORA_R,                     # LoRA秩
    lora_alpha=LORA_ALPHA,        # LoRA缩放参数
    lora_dropout=LORA_DROPOUT,    # LoRA Dropout
    target_modules=LORA_TARGET_MODULES,  # 目标模块
    bias="none",                  # 不训练bias
    inference_mode=False          # 训练模式
)

# 将LoRA应用到模型
model = get_peft_model(model, lora_config)

# 打印模型信息
model.print_trainable_parameters()

print("\n✅ LoRA配置完成")
print(f"   - LoRA参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"   - 参数效率: {sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters()) * 100:.2f}%")

Some weights of the model checkpoint at ../bert-base-chinese were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint

✅ BERT基础模型加载完成
   - 原始模型参数数量: 102,269,186
trainable params: 297988 || all params: 102565636 || trainable%: 0.29053395622682043

✅ LoRA配置完成
   - LoRA参数数量: 297,988
   - 参数效率: 0.29%


In [9]:
"""
==========================================
第八部分：准备训练数据
==========================================
加载训练集和验证集，并创建DataLoader
"""

# 加载数据集
train_dataset = GenerateData(mode='train')
val_dataset = GenerateData(mode='val')

# 创建DataLoader
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_dataloader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE
)

print("\n✅ 训练数据准备完成")
print(f"   - 训练批次数: {len(train_dataloader)}")
print(f"   - 验证批次数: {len(val_dataloader)}")

数据基本信息
数据总量: 11987

标签分布:
0    7987
1    4000
Name: label, dtype: int64

数据示例（前3条）:
   label        review
0      1  很快，好吃，味道足，量大
1      1  没有送水没有送水没有送水
2      1      非常快，态度好。

数据分割结果
训练集大小: 8390 (70.0%)
验证集大小: 1798 (15.0%)
测试集大小: 1799 (15.0%)

✅ 数据加载和预处理完成

✅ 训练数据准备完成
   - 训练批次数: 525
   - 验证批次数: 113


In [ ]:
"""
==========================================
第九部分：配置训练环境
==========================================
设置设备、损失函数和优化器
"""

# 设备配置
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f"使用设备: {device}")

# 损失函数（BertForSequenceClassification内部已包含损失函数）
# 但我们也可以单独定义用于验证
criterion = nn.CrossEntropyLoss()

# AdamW优化器
# 注意：LoRA通常使用更高的学习率
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eps=1e-8
)

# 将模型移到指定设备
if use_cuda:
    model = model.cuda()
    criterion = criterion.cuda()

print("✅ 训练环境配置完成")

In [ ]:
"""
==========================================
第十部分：模型保存函数
==========================================
定义LoRA模型保存函数
"""

def save_lora_model(model, save_name):
    """
    保存LoRA模型权重
    
    参数:
        model: PEFT模型对象
        save_name: 保存的文件夹名（如 'best' 或 'last'）
    """
    # 创建保存路径
    save_dir = os.path.join(SAVE_PATH, save_name)
    
    # 保存LoRA权重（只保存可训练的LoRA参数）/这个函数只保存被修改的参数，而被冻结的参数不会修改所以bert没被保存，但是Lorag
    model.save_pretrained(save_dir)
    
    print(f"✅ LoRA模型已保存: {save_dir}")
    print(f"   💡 提示：LoRA权重文件很小（通常<10MB），而完整BERT模型约400MB")

print("✅ 模型保存函数定义完成")

In [ ]:
"""
==========================================
第十一部分：训练模型
==========================================
使用LoRA进行高效微调
"""

print("\n开始训练...")
print("=" * 50)

# 初始化最佳验证准确率
best_dev_acc = 0

# 记录训练开始时间
train_start_time = time.time()

for epoch_num in range(EPOCHS):
    # ========== 训练阶段 ==========
    model.train()
    
    total_acc_train = 0
    total_loss_train = 0
    
    # 遍历训练数据批次
    for train_input, train_label in tqdm(
        train_dataloader, 
        desc=f"Epoch {epoch_num + 1}/{EPOCHS} [训练]"
    ):
        # 将数据移到指定设备
        train_label = train_label.to(device)
        mask = train_input['attention_mask'].to(device)
        input_id = train_input['input_ids'].squeeze(1).to(device)
        
        # 前向传播
        # BertForSequenceClassification 返回包含loss和logits的对象
        outputs = model(
            input_ids=input_id,
            attention_mask=mask,
            labels=train_label
        )
        
        batch_loss = outputs.loss
        logits = outputs.logits
        
        total_loss_train += batch_loss.item()
        
        # 计算准确率
        acc = (logits.argmax(dim=1) == train_label).sum().item()
        total_acc_train += acc
        
        # 反向传播和参数更新
        model.zero_grad()
        batch_loss.backward()
        optimizer.step()
    
    # ========== 验证阶段 ==========
    model.eval()
    
    total_acc_val = 0
    total_loss_val = 0
    
    with torch.no_grad():
        for val_input, val_label in tqdm(
            val_dataloader,
            desc=f"Epoch {epoch_num + 1}/{EPOCHS} [验证]"
        ):
            val_label = val_label.to(device)
            mask = val_input['attention_mask'].to(device)
            input_id = val_input['input_ids'].squeeze(1).to(device)
            
            outputs = model(
                input_ids=input_id,
                attention_mask=mask,
                labels=val_label
            )
            
            batch_loss = outputs.loss
            logits = outputs.logits
            
            total_loss_val += batch_loss.item()
            
            acc = (logits.argmax(dim=1) == val_label).sum().item()
            total_acc_val += acc
    
    # ========== 输出训练结果 ==========
    val_accuracy = total_acc_val / len(val_dataset)
    print(
        f'''\nEpochs: {epoch_num + 1} 
          | Train Loss: {total_loss_train / len(train_dataset): .3f} 
          | Train Accuracy: {total_acc_train / len(train_dataset): .3f} 
          | Val Loss: {total_loss_val / len(val_dataset): .3f} 
          | Val Accuracy: {val_accuracy: .3f}'''
    )
    
    # ========== 保存最佳模型 ==========
    if val_accuracy > best_dev_acc:
        best_dev_acc = val_accuracy
        save_lora_model(model, 'best')
        print(f"   🎯 发现更好的模型！验证准确率: {best_dev_acc:.3f}")
    
    print("=" * 50)

# 记录训练结束时间
train_end_time = time.time()
training_time_sec = train_end_time - train_start_time
training_time_min = training_time_sec / 60

# 保存最后一个epoch的模型
save_lora_model(model, 'last')

print("\n✅ 训练完成！")
print(f"   - 最佳验证准确率: {best_dev_acc:.3f}")
print(f"   - 最佳模型已保存: {os.path.join(SAVE_PATH, 'best')}")
print(f"   - 最后模型已保存: {os.path.join(SAVE_PATH, 'last')}")

In [ ]:
"""
==========================================
第十二部分：在测试集上评估模型
==========================================
使用训练好的LoRA模型在测试集上进行最终评估
"""

# 准备测试数据
test_dataset = GenerateData(mode='test')
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# 评估模型
model.eval()

# 初始化累计指标
total_acc_test = 0
total_loss_test = 0

# 记录真实标签和预测标签，用于计算指标
all_labels = []
all_preds = []

with torch.no_grad():
    for test_input, test_label in tqdm(test_dataloader, desc="测试中"):
        test_label = test_label.to(device)
        mask = test_input['attention_mask'].to(device)
        input_id = test_input['input_ids'].squeeze(1).to(device)
        
        outputs = model(
            input_ids=input_id,
            attention_mask=mask,
            labels=test_label
        )
        
        batch_loss = outputs.loss
        logits = outputs.logits
        
        total_loss_test += batch_loss.item()
        
        preds = logits.argmax(dim=1)
        acc = (preds == test_label).sum().item()
        total_acc_test += acc

        # 保存标签与预测
        all_labels.extend(test_label.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

# 输出评估结果
avg_loss = total_loss_test / len(test_dataset)
avg_accuracy = total_acc_test / len(test_dataset)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_preds,
    average='binary',
    zero_division=0
)

In [ ]:

print(f"\n测试集评估结果:")
print(f"  - Loss: {avg_loss: .3f}")
print(f"  - Accuracy: {avg_accuracy: .3f}")
print(f"  - Precision: {precision: .3f}")
print(f"  - Recall: {recall: .3f}")
print(f"  - F1 Score: {f1: .3f}")
print(f"   - 训练时间: {training_time_sec:.1f} 秒 ({training_time_min:.2f} 分钟)")

print(f"\n🎉 最终测试准确率: {avg_accuracy:.3f}")